In [4]:
import pandas as pd
import numpy as np
ext_raw = pd.read_csv("loan_extensions.csv")
ext_raw.head()

,extension_id,loan_number,extension_sequence,extension_request_date,extension_days,extension_fee_percent,extension_fee_amount,requested_by_role,approved_by_employee_id,reason_code,lock_expiration_date_before,lock_expiration_date_after
0,90001,70000004,1,2023-05-02,15,0.00250,1910.11,Processor,126,Title Issues,2023-05-04,2023-05-19
1,90002,70000004,2,2023-05-15,5,0.00125,955.06,Loan Officer,125,Appraisal Delay,2023-05-19,2023-05-24
2,90003,70000007,1,2023-04-22,15,0.00250,1189.77,Broker,128,Title Issues,2023-04-27,2023-05-12
3,90004,70000007,2,2023-05-10,15,0.00250,1189.77,Broker,138,Title Issues,2023-05-12,2023-05-27
4,90005,70000009,1,2023-05-10,15,0.00250,1043.73,Loan Officer,128,Appraisal Delay,2023-05-15,2023-05-30


In [6]:
ext_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4069 entries, 0 to 4068
Data columns (total 12 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   extension_id                 4069 non-null   int64  
 1   loan_number                  4069 non-null   int64  
 2   extension_sequence           4069 non-null   int64  
 3   extension_request_date       4069 non-null   object 
 4   extension_days               4069 non-null   int64  
 5   extension_fee_percent        4069 non-null   float64
 6   extension_fee_amount         4069 non-null   float64
 7   requested_by_role            4069 non-null   object 
 8   approved_by_employee_id      4069 non-null   int64  
 9   reason_code                  4069 non-null   object 
 10  lock_expiration_date_before  4069 non-null   object 
 11  lock_expiration_date_after   4069 non-null   object 
dtypes: float64(2), int64(5), object(5)
memory usage: 381.6+ KB


In [7]:
Extensions = ext_raw.rename(columns={
    "extension_id": "Extension ID",
    "loan_number": "Loan Number",
    "extension_sequence": "Extension Number",
    "extension_request_date": "Date Extension Requested",
    "extension_days": "Extension (Days)",
    "extension_fee_percent": "Extension Cost (%)",
    "extension_fee_amount": "Extension Cost ($)",
    "requested_by_role": "Extension Requestor",
    "approved_by_employee_id": "Approved By (ID)",
    "reason_code": "Extension Reason",
    "lock_expiration_date_before": "Expiration Before Extension",
    "lock_expiration_date_after": "Expiration After Extension",
})

In [9]:
ext_clean = Extensions.copy()

In [10]:
ext_clean.head()

,Extension ID,Loan Number,Extension Number,Date Extension Requested,Extension (Days),Extension Cost (%),Extension Cost ($),Extension Requestor,Approved By (ID),Extension Reason,Expiration Before Extension,Expiration After Extension
0,90001,70000004,1,2023-05-02,15,0.00250,1910.11,Processor,126,Title Issues,2023-05-04,2023-05-19
1,90002,70000004,2,2023-05-15,5,0.00125,955.06,Loan Officer,125,Appraisal Delay,2023-05-19,2023-05-24
2,90003,70000007,1,2023-04-22,15,0.00250,1189.77,Broker,128,Title Issues,2023-04-27,2023-05-12
3,90004,70000007,2,2023-05-10,15,0.00250,1189.77,Broker,138,Title Issues,2023-05-12,2023-05-27
4,90005,70000009,1,2023-05-10,15,0.00250,1043.73,Loan Officer,128,Appraisal Delay,2023-05-15,2023-05-30


In [12]:
date_cols = [
    "Date Extension Requested",
    "Expiration Before Extension",
    "Expiration After Extension"
]

for col in date_cols:
    ext_clean[col] = pd.to_datetime(ext_clean[col], errors="coerce")

In [13]:
numeric_cols = [
    "Extension (Days)",
    "Extension Cost (%)",
    "Extension Cost ($)",
    "Extension Number"
]

for col in numeric_cols:
    ext_clean[col] = pd.to_numeric(ext_clean[col], errors="coerce")

In [14]:
text_cols = ["Extension Requestor", "Extension Reason"]

for col in text_cols:
    ext_clean[col] = (
        ext_clean[col]
        .astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .str.title()
    )

In [15]:
ext_clean["Total Extension Days"] = (
    ext_clean.groupby("Loan Number")["Extension (Days)"].transform("sum")
)

In [16]:
ext_clean["Total Extension Cost ($)"] = (
    ext_clean.groupby("Loan Number")["Extension Cost ($)"].transform("sum")
)

In [17]:
ext_clean["Extension Count"] = (
    ext_clean.groupby("Loan Number")["Extension Number"].transform("max")
)

In [18]:
ext_clean.duplicated().sum()

0

In [19]:
ext_clean.duplicated(subset=["Loan Number", "Extension Number"]).sum()

0

In [20]:
ext_clean.isna().sum()

Extension ID                   0
Loan Number                    0
Extension Number               0
Date Extension Requested       0
Extension (Days)               0
Extension Cost (%)             0
Extension Cost ($)             0
Extension Requestor            0
Approved By (ID)               0
Extension Reason               0
Expiration Before Extension    0
Expiration After Extension     0
Total Extension Days           0
Total Extension Cost ($)       0
Extension Count                0
dtype: int64

In [21]:
ext_clean["Extension Bucket"] = pd.cut(
    ext_clean["Extension Count"],
    bins=[-1, 0, 2, 100],
    labels=["0 Extensions", "1–2 Extensions", "3+ Extensions"]
)

In [22]:
ext_clean.head()

,Extension ID,Loan Number,Extension Number,Date Extension Requested,Extension (Days),Extension Cost (%),Extension Cost ($),Extension Requestor,Approved By (ID),Extension Reason,Expiration Before Extension,Expiration After Extension,Total Extension Days,Total Extension Cost ($),Extension Count,Extension Bucket
0,90001,70000004,1,2023-05-02,15,0.00250,1910.11,Processor,126,Title Issues,2023-05-04,2023-05-19,20,2865.17,2,1–2 Extensions
1,90002,70000004,2,2023-05-15,5,0.00125,955.06,Loan Officer,125,Appraisal Delay,2023-05-19,2023-05-24,20,2865.17,2,1–2 Extensions
2,90003,70000007,1,2023-04-22,15,0.00250,1189.77,Broker,128,Title Issues,2023-04-27,2023-05-12,30,2379.54,2,1–2 Extensions
3,90004,70000007,2,2023-05-10,15,0.00250,1189.77,Broker,138,Title Issues,2023-05-12,2023-05-27,30,2379.54,2,1–2 Extensions
4,90005,70000009,1,2023-05-10,15,0.00250,1043.73,Loan Officer,128,Appraisal Delay,2023-05-15,2023-05-30,30,2087.46,2,1–2 Extensions


In [24]:
ext_clean["Extension Reason"].value_counts()

Extension Reason
Borrower Delay             844
Rate Shopping              827
Appraisal Delay            813
Title Issues               797
Underwriting Conditions    788
Name: count, dtype: int64

In [26]:
ext_clean["Extension Count"].value_counts()

Extension Count
2    1662
1    1511
3     456
4     440
Name: count, dtype: int64

In [28]:
ext_clean.to_csv("ext_clean.csv", index=False)